# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring — which pages should editors review first for refresh, expansion, protection, pruning, or monitoring?

This notebook walks through the data contract for my lane: what a row means, which tables and windows I use, the five features I build, the leakage trap I spring on myself, and the honest limitation I accept.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the warehouse release

Iterate on a **mid-panel month** (2026-03), not the final June month. The final month is sealed as the natural outcome window. Use the `_sample` table (which is June 2026) only to test query mechanics, never to develop label logic.

For local runs without `HF_TOKEN`, we fall back to the starter CSV so the notebook still executes top to bottom. For the real numbers, paste a Hugging Face READ token when prompted.

In [1]:
import os, sys, getpass
import pandas as pd
import numpy as np
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
HF_TOKEN = os.environ.get("HF_TOKEN")

# ── Try DuckDB + Hugging Face first ──────────────────────────────────────
USE_WAREHOUSE = False
con = None
TABLES = None

if HF_TOKEN or IN_COLAB:
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
    try:
        import duckdb
        con = duckdb.connect()
        con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
        REL = "hf://datasets/FlyRank/internship-warehouse"
        TABLES = {
            "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
            "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
            "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
            "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
            "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
        }
        USE_WAREHOUSE = True
        print("✓ Connected to Hugging Face warehouse release via DuckDB")
    except Exception as e:
        print(f"⚠ Warehouse unavailable ({e.__class__.__name__}: {e}). Falling back to starter CSV.")

# ── Fallback: starter CSV ────────────────────────────────────────────────
starter_raw = None
if not USE_WAREHOUSE:
    relative_data_path = Path("data") / "raw" / "content_refresh_anonymized.csv"
    repo_root = None
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_data_path
        if candidate.exists():
            repo_root = base
            data_path = candidate
            break
    if repo_root is None:
        raise FileNotFoundError(f"Could not find starter CSV or Hugging Face data")
    starter_raw = pd.read_csv(data_path)
    print(f"✓ Using starter CSV fallback: {len(starter_raw):,} rows from {data_path}")

# ── Helper: DuckDB query → pandas DataFrame ──────────────────────────────
def q(sql: str) -> pd.DataFrame:
    if con is None:
        raise RuntimeError("DuckDB not connected")
    return con.sql(sql).df()


✓ Using starter CSV fallback: 30,000 rows from C:\Users\Huniza Computer\Downloads\Mustafa-FlyRank-main\Mustafa-FlyRank\data\raw\content_refresh_anonymized.csv


## 1. The contract — five plain-words answers

**1. What one row means for my lane:**
> One row = **one content item (one page)**, aggregated to a decision point at the close of a mid-panel month. My model scores pages, not page-days, so the editor sees each page once in the review queue — I roll the daily fact up to the content level using time-window features.

**2. Which table(s) I'll use:**
> `fact_content_daily_performance` (the 79M-row daily grain, partitioned by month — I use month=`2026-03` for iteration) joined to `dim_content` on `content_hash_id` for keyword/metadata context, plus `dim_clients` on `client_hash_id` only to filter clients whose GA4 tracking started before my feature window.

**3. Which time window:**
> **Feature window** = the 60 days *before* 2026-03-31 (i.e. 2026-02-01 → 2026-03-31, split into last-30 and prev-30). **Label window** = the 30 days *after* 2026-03-31 (i.e. 2026-04-01 → 2026-04-30). Feature and label windows are disjoint. For the mid-panel iteration I build features from the 60 days ending 2026-03-31 and the label from April 2026.

**4. What I'd predict or rank (label or proxy):**
> **Label: `is_declining_next_30d`** — 1 if a page's impressions drop by ≥20% in the label window (April 2026) compared to the prev-30 of the feature window (2026-02-01 → 2026-02-28), requiring ≥100 prev-30 impressions to avoid calling noise a decline. Primary output is a **ranked score per page** (not a hard classify) so editors can work top-K first.

**5. One thing I deliberately exclude:**
> I exclude **any field computed from the label window or any field that encodes the trend bucket** — specifically `trend_pct`, `trend_direction`, and any `_last30` column from the label month itself. I also exclude LLM provider/model columns (`provider_used`, `model_used`) because they record the *generator*, not the *page's behaviour* — they would let the model copy a past editorial decision rather than learn a page signal.

In [2]:
# Printed confirmation of the five contract answers (so a quick scan of the cell outputs catches them too)
print("Contract summary for Refresh / Content Opportunity Scoring lane:")
print("  1 row        = one content_item page (aggregated), not a page-day")
print("  Tables       = fact_content_daily_performance + dim_content (+ dim_clients for GA4 start)")
print("  Feature win. = 60d ending 2026-03-31 (last30 = Mar, prev30 = Feb)")
print("  Label win.   = 30d after 2026-03-31  (April 2026) — disjoint from features")
print("  Label        = is_declining_next_30d (Apr impressions < 80% of Feb prev30, prev30 ≥ 100 imp)")
print("  Excluded     = trend_pct / trend_direction / any label-window field / LLM provider cols")

Contract summary for Refresh / Content Opportunity Scoring lane:
  1 row        = one content_item page (aggregated), not a page-day
  Tables       = fact_content_daily_performance + dim_content (+ dim_clients for GA4 start)
  Feature win. = 60d ending 2026-03-31 (last30 = Mar, prev30 = Feb)
  Label win.   = 30d after 2026-03-31  (April 2026) — disjoint from features
  Label        = is_declining_next_30d (Apr impressions < 80% of Feb prev30, prev30 ≥ 100 imp)
  Excluded     = trend_pct / trend_direction / any label-window field / LLM provider cols


## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket. The rule of thumb for a feature: **would I have known this value on the morning of 2026-04-01, before April traffic existed?**

In [3]:
field_buckets = pd.DataFrame([
    # ── FEATURES (knowable 2026-04-01 morning, before April data exists) ──
    ("imp_prev30",            "feature", "GSC impressions 2026-02-01 → 2026-02-28 — volume baseline"),
    ("clk_prev30",            "feature", "GSC clicks in prev30 — demand captured"),
    ("pos_avg_prev30",        "feature", "Mean GSC avg position in prev30 — visibility rank"),
    ("sessions_prev30",       "feature", "GA4 sessions in prev30 — engagement volume baseline"),
    ("days_with_imp_60d",     "feature", "Days with ≥1 impression across the full 60d feature window"),
    ("ctr_prev30",            "feature", "clk_prev30 / imp_prev30 × 100 — click capture per impression"),
    ("imp_mom_ratio_last30_prev30", "feature", "(Mar imp / Feb imp) — momentum the month *before* the label month"),
    ("content_age_at_decision", "feature", "Days from content creation to 2026-03-31 — from dim_content"),
    ("word_count",            "feature", "Article word count — from dim_content (content property, not outcome)"),
    ("search_volume",         "feature", "Keyword search-volume estimate — static keyword metadata from dim_content"),

    # ── LABEL / PROXY (the thing to predict, never a feature) ─────────────
    ("is_declining_next_30d", "label",   "1 if April imp < 0.8 × February imp AND February imp ≥ 100; else 0"),
    ("imp_next30",            "label",   "Absolute impressions in the April 2026 label window — raw label numerator"),

    # ── CONTEXT (grouping / joining / splitting, never learned from) ─────
    ("content_hash_id",       "context", "Pseudonymous page id — joins, dedup, grouped review only"),
    ("client_hash_id",        "context", "Pseudonymous client id — client-holdout splits and grouped checks"),
    ("content_type",          "context", "Category used in breakdowns and case reading; not passed to the model itself"),
    ("keyword_hash_id",       "context", "Keyword group id for cannibalization checks later; not a model input"),
    ("ga4_data_available",    "context", "Flag used to FILTER rows (drop GSC-only rows from prev30); not a feature"),

    # ── EXCLUDED (each gets a why) ────────────────────────────────────────
    ("trend_pct",             "excluded", "Derived from the same last-vs-prev comparison the label IS — direct answer leak"),
    ("trend_direction",       "excluded", "Bucket computed from trend_pct — is the label bucket itself for any same-window definition"),
    ("imp_last30_OF_APRIL",   "excluded", "Label-window impressions — by definition not known on 2026-04-01 morning"),
    ("provider_used",         "excluded", "Records which LLM wrote the page; lets model copy editorial choices, not learn page behaviour"),
    ("model_used",            "excluded", "Same as provider_used: product/process metadata, not a content-performance signal"),
    ("fact_query_90d.*impressions_90d", "excluded", "The 90d query table's window overlaps the label month — its totals contain April unless I restrict to _prev30; kept out of v1 for safety"),
], columns=["field", "bucket", "notes"])

print("Field classification — rows per bucket:")
print(field_buckets["bucket"].value_counts().to_string())
print()
for b in ["feature", "label", "context", "excluded"]:
    sub = field_buckets[field_buckets["bucket"] == b]
    print(f"── {b.upper()} ({len(sub)}) ──")
    for _, r in sub.iterrows():
        print(f"  {r['field']:<32} {r['notes']}")
    print()

Field classification — rows per bucket:
bucket
feature     10
excluded     6
context      5
label        2

── FEATURE (10) ──
  imp_prev30                       GSC impressions 2026-02-01 → 2026-02-28 — volume baseline
  clk_prev30                       GSC clicks in prev30 — demand captured
  pos_avg_prev30                   Mean GSC avg position in prev30 — visibility rank
  sessions_prev30                  GA4 sessions in prev30 — engagement volume baseline
  days_with_imp_60d                Days with ≥1 impression across the full 60d feature window
  ctr_prev30                       clk_prev30 / imp_prev30 × 100 — click capture per impression
  imp_mom_ratio_last30_prev30      (Mar imp / Feb imp) — momentum the month *before* the label month
  content_age_at_decision          Days from content creation to 2026-03-31 — from dim_content
  word_count                       Article word count — from dim_content (content property, not outcome)
  search_volume                    Keyword 

## 3. Verify it with queries (month=2026-03)

Every contract claim above gets a query cell here. A contract claim without a query next to it is a guess. I run **exactly three verification queries** on the mid-panel month: (1) the grain holds, (2) row count and date span of my slice, and (3) availability — how many rows survive the `IS TRUE` filter on GA4 availability.

### Verification query 1: Grain — one row really is (report_date × client_hash_id × content_hash_id)

The daily fact claims `report_date × client × content` is the grain. If my GROUP BY on those three columns returns *any* row with `COUNT(*) > 1`, the grain is a lie. Zero rows back = grain holds.

In [4]:
if USE_WAREHOUSE:
    q1 = q(f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
        LIMIT 5
    """)
    print(f"Grain probe rows with duplicate (report_date × client × content) in month=2026-03: {len(q1)}")
    if len(q1):
        display(q1)
    else:
        print("  ✓ Grain holds: zero duplicates found. One row = one report_date × client × content.")
else:
    # Starter CSV is one row per content_id — verify that grain instead
    dupes = starter_raw.groupby(["content_id"]).size().rename("c").reset_index()
    dupes = dupes[dupes["c"] > 1]
    print(f"Starter CSV grain probe — rows with duplicate content_id: {len(dupes)}")
    if len(dupes):
        display(dupes.head())
    else:
        print("  ✓ Starter grain holds: one row = one content_id.")

Starter CSV grain probe — rows with duplicate content_id: 0
  ✓ Starter grain holds: one row = one content_id.


### Verification query 2: Row count + date span of my lane's slice in month=2026-03

My lane slice for month=2026-03 = content items that appear in the daily fact for that month, from clients whose GA4 tracking started before 2026-02-01 (so prev30 GA4 rows are real, not zero-filled). Print how many (date × client × content) rows that is, the MIN/MAX report_date, and how many distinct content items that rolls up to — because my final decision grain is one row per content item.

In [5]:
if USE_WAREHOUSE:
    q2 = q(f"""
        SELECT
          COUNT(*)                                                   AS daily_rows,
          MIN(report_date)                                           AS min_date,
          MAX(report_date)                                           AS max_date,
          COUNT(DISTINCT client_hash_id)                             AS n_clients,
          COUNT(DISTINCT content_hash_id)                            AS n_content_items
        FROM {TABLES['fact_daily']} f
        INNER JOIN {TABLES['dim_clients']} c
          ON f.client_hash_id = c.client_hash_id
        WHERE f.month = '2026-03'
          AND c.ga4_data_start <= DATE '2026-02-01'
    """)
    display(q2.style.set_caption("Month=2026-03 slice after GA4 start filter"))
    r = q2.iloc[0]
    print(f"{r['daily_rows']:,} daily rows span {r['min_date']} → {r['max_date']} "
          f"across {r['n_clients']} clients and {r['n_content_items']:,} distinct content items.")
else:
    # Starter CSV slice: rows with impressions>0 and age≥90 (per the starter pipeline)
    lane_slice = starter_raw[(starter_raw["impressions_90d"] > 0) & (starter_raw["content_age_days"] >= 90)].copy()
    summary = pd.DataFrame([{
        "rows":              len(lane_slice),
        "n_clients":         lane_slice["client_id"].nunique(),
        "n_content_items":   lane_slice["content_id"].nunique(),
        "trailing_window":   "90d at export time (starter slice is a point-in-time snapshot)",
        "declining_proxy":   (lane_slice["trend_direction"].str.lower() == "down").mean(),
    }])
    display(summary.style.set_caption("Starter CSV slice (fallback): rows with imp>0 & age≥90"))
    r = summary.iloc[0]
    print(f"{r['rows']:,} content rows across {r['n_clients']} clients, declining proxy = {r['declining_proxy']:.1%}.")

,rows,n_clients,n_content_items,trailing_window,declining_proxy
0,30000,32,30000,90d at export time (starter slice is a point-in-time snapshot),0.542067


30,000 content rows across 32 clients, declining proxy = 54.2%.


### Verification query 3: Availability — filter with `IS TRUE` and show how many rows survive

GA4 columns are zero-filled before each client's GA4 onboarding date, with `ga4_data_available = FALSE` on those rows. Treating those zeros as "no engagement" is a classic trap. I require `ga4_data_available IS TRUE` for any daily row whose session/engagement numbers I use in the prev30 window. Show before → after counts in month=2026-03.

In [6]:
if USE_WAREHOUSE:
    q3 = q(f"""
        WITH all_rows AS (
            SELECT COUNT(*) AS c_total,
                   COUNT(*) FILTER (WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28') AS c_prev30
            FROM {TABLES['fact_daily']} f
            INNER JOIN {TABLES['dim_clients']} cc
              ON f.client_hash_id = cc.client_hash_id
            WHERE f.month IN ('2026-02', '2026-03')
              AND cc.ga4_data_start <= DATE '2026-02-01'
        ),
        filtered AS (
            SELECT COUNT(*) AS c_total_survive,
                   COUNT(*) FILTER (WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28') AS c_prev30_survive
            FROM {TABLES['fact_daily']} f
            INNER JOIN {TABLES['dim_clients']} cc
              ON f.client_hash_id = cc.client_hash_id
            WHERE f.month IN ('2026-02', '2026-03')
              AND cc.ga4_data_start <= DATE '2026-02-01'
              AND f.ga4_data_available IS TRUE
        )
        SELECT
          a.c_total,              f.c_total_survive,
          ROUND(100.0 * f.c_total_survive    / NULLIF(a.c_total,    0), 2) AS pct_survive_total,
          a.c_prev30,             f.c_prev30_survive,
          ROUND(100.0 * f.c_prev30_survive   / NULLIF(a.c_prev30,   0), 2) AS pct_survive_prev30
        FROM all_rows a, filtered f
    """)
    display(q3.style.set_caption("Availability: ga4_data_available IS TRUE (prev30 = Feb 2026)"))
    r = q3.iloc[0]
    print(f"Full Feb+Mar daily rows: {r['c_total']:,}. "
          f"Survive ga4_data_available IS TRUE: {r['c_total_survive']:,} = {r['pct_survive_total']:.1f}%.")
    print(f"Prev30 (Feb) rows:      {r['c_prev30']:,}. "
          f"Survive ga4_data_available IS TRUE: {r['c_prev30_survive']:,} = {r['pct_survive_prev30']:.1f}%.")
    print("  → This confirms I don't silently read zero-filled GA4 columns as 'no engagement'.")
else:
    # Starter CSV: the analogous availability check is 'do the engagement columns actually have data?'
    lane = starter_raw[(starter_raw["impressions_90d"] > 0) & (starter_raw["content_age_days"] >= 90)].copy()
    has_sessions = (lane["sessions_90d"] > 0).sum()
    has_sessions_and_imp = (lane["sessions_90d"] > 0) & (lane["impressions_90d"] > 0)
    # The 'IS TRUE' analogue in the starter: engagement_rate is computable only when sessions_90d > 0
    availability = pd.DataFrame([{
        "rows_in_slice":                    len(lane),
        "rows_with_sessions_90d":           int(has_sessions),
        "pct_with_sessions":                f"{100*has_sessions/len(lane):.1f}%",
        "rows_where_engagement_rate_IS_NOT_null": int(lane["engagement_rate"].notna().sum()),
        "pct_engagement_rate_nonnull":      f"{100*lane['engagement_rate'].notna().mean():.1f}%",
    }])
    display(availability.style.set_caption("Starter availability analogue: sessions_90d > 0 (the 'IS TRUE' check)") )
    print(f"{availability.iloc[0]['pct_engagement_rate_nonnull']} of rows have a measurable engagement_rate; "
          "the rest I mark as 'no engagement data' rather than filling with 0.")

,rows_in_slice,rows_with_sessions_90d,pct_with_sessions,rows_where_engagement_rate_IS_NOT_null,pct_engagement_rate_nonnull
0,30000,30000,100.0%,30000,100.0%


100.0% of rows have a measurable engagement_rate; the rest I mark as 'no engagement data' rather than filling with 0.


## 3b. Five features (max) + the deliberate leakage trap

I build a **small feature frame per content item** from the same mid-panel (month=2026-03) contract. Five features only. Each gets a one-line *knowable at the decision moment because…* justification.

Then I spring the trap everyone falls into: add ONE label-derived column by mistake, watch the score jump near 1.00, then DELETE it and keep the honest number — the notebook-02 lesson, performed on real warehouse data by me.

In [7]:
if USE_WAREHOUSE:
    feature_frame = q(f"""
        WITH daily AS (
            SELECT f.*
            FROM {TABLES['fact_daily']} f
            INNER JOIN {TABLES['dim_clients']} c
              ON f.client_hash_id = c.client_hash_id
            WHERE c.ga4_data_start <= DATE '2026-02-01'
              AND f.report_date BETWEEN DATE '2026-02-01' AND DATE '2026-04-30'
              -- keep only GSC-available rows for impression aggregations
              AND f.gsc_data_available IS TRUE
        ),
        agg AS (
            SELECT client_hash_id, content_hash_id,
              -- prev30  = 2026-02-01 .. 2026-02-28 (feature baseline)
              SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                       THEN f.gsc_impressions END)                                                              AS imp_prev30,
              SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                       THEN f.gsc_clicks END)                                                                     AS clk_prev30,
              AVG(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                       THEN f.gsc_avg_position END)                                                                AS pos_avg_prev30,
              SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                            AND f.ga4_data_available IS TRUE
                       THEN f.ga4_sessions END)                                                                   AS sessions_prev30,
              -- last30  = 2026-03-01 .. 2026-03-31 (pre-label momentum — still FEATURE window)
              SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
                       THEN f.gsc_impressions END)                                                              AS imp_last30_featurewin,
              -- 60-day activity span
              COUNT(DISTINCT CASE WHEN f.gsc_impressions >= 1 THEN report_date END)                              AS days_with_imp_60d,
              -- next30  = 2026-04-01 .. 2026-04-30 (LABEL window — never a feature)
              SUM(CASE WHEN report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
                       THEN f.gsc_impressions END)                                                              AS imp_next30
            FROM daily f
            GROUP BY 1, 2
        ),
        final AS (
            SELECT a.*,
                   dc.word_count, dc.search_volume,
                   DATE_DIFF(DATE '2026-03-31', dc.content_created_at, DAY)                         AS content_age_at_decision,
                   -- derived features
                   ROUND(100.0 * clk_prev30 / NULLIF(imp_prev30, 0), 3)                              AS ctr_prev30,
                   ROUND(1.0 * imp_last30_featurewin / NULLIF(imp_prev30, 0), 4)                     AS imp_mom_ratio_last30_prev30,
                   -- label
                   CASE WHEN imp_prev30 >= 100
                             AND imp_next30 < 0.8 * imp_prev30
                        THEN 1 ELSE 0 END                                                           AS is_declining_next_30d
            FROM agg a
            LEFT JOIN {TABLES['dim_content']} dc
              ON a.content_hash_id = dc.content_hash_id
            WHERE imp_prev30 >= 100     -- enough baseline volume to define decline honestly
        )
        SELECT * FROM final
    """)
else:
    # Fallback: build a feature frame from the starter CSV using its 90d/30d pre-built columns.
    lane = starter_raw[(starter_raw["impressions_90d"] > 0) & (starter_raw["content_age_days"] >= 90)].copy()
    # Feature window analogue: use *_prev_30d as the baseline, *_last_30d as the "momentum before label".
    # Label proxy: the starter ships trend_direction from trend_pct — treat the 'down' bucket as our label proxy.
    feature_frame = pd.DataFrame({
        "content_id":                     lane["content_id"],
        "client_id":                      lane["client_id"],
        "imp_prev30":                     lane["impressions_prev_30d"],
        "clk_prev30":                     lane["clicks_prev_30d"],
        "pos_avg_prev30":                 lane["avg_position"].replace(0, np.nan),  # starter avg_position is 90d; 0 means 'no data'
        "sessions_prev30":                lane["sessions_prev_30d"],
        "days_with_imp_60d":              lane["days_with_impressions"],
        "imp_last30_featurewin":          lane["impressions_last_30d"],
        "imp_next30":                     np.nan,  # starter has no forward window — leave NaN
        "word_count":                     lane["word_count"],
        "search_volume":                  lane["search_volume"],
        "content_age_at_decision":        lane["content_age_days"],
    })
    feature_frame["ctr_prev30"] = (100.0 * feature_frame["clk_prev30"] / feature_frame["imp_prev30"].replace(0, np.nan)).round(3)
    feature_frame["imp_mom_ratio_last30_prev30"] = (
        feature_frame["imp_last30_featurewin"] / feature_frame["imp_prev30"].replace(0, np.nan)
    ).round(4)
    feature_frame["is_declining_next_30d"] = (lane["trend_direction"].str.lower() == "down").astype(int).values
    feature_frame = feature_frame[feature_frame["imp_prev30"] >= 100].reset_index(drop=True)

# ── Choose exactly five features (no more!) and show the frame ───────────
# Five honest features. NOTE: imp_mom_ratio_last30_prev30 is safe on the
# warehouse (Mar/Feb both before the April label window) but on the starter CSV
# its equivalent directly computes the starter proxy label (same-window trap).
# We use word_count here so the honest-vs-leak contrast is visible in both paths.
FIVE_FEATURES = [
    "imp_prev30",
    "ctr_prev30",
    "word_count",
    "content_age_at_decision",
    "days_with_imp_60d",
]
LABEL = "is_declining_next_30d"

feats = feature_frame[FIVE_FEATURES + [LABEL]].copy()
# Fill numeric blanks with 0 only where 'no data' genuinely means 0 (counts); keep NaN-sentinel policy for rates via clip
feats[FIVE_FEATURES] = feats[FIVE_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Feature frame: {len(feats):,} content items × {len(FIVE_FEATURES)} features + 1 label")
print(f"Base rate P({LABEL}=1) = {feats[LABEL].mean():.1%}")
print()
print("Feature 1: imp_prev30")
print("  knowable at the decision moment because it sums Feb 2026 GSC impressions (prev30),")
print("  which were measured and finalized before April 1 — my label hasn't started yet.")
print()
print("Feature 2: ctr_prev30")
print("  knowable at the decision moment because it's the ratio of prev30 clicks over prev30")
print("  impressions — both closed by Feb 28; no April data participates.")
print()
print("Feature 3: word_count")
print("  knowable at the decision moment because it's a static content property from dim_content")
print("  — measured when the article was created, never changes, never references the label month.")
print()
print("Feature 4: content_age_at_decision")
print("  knowable at the decision moment because it's (2026-03-31 minus content_created_at).")
print("  Created-at is static metadata from dim_content; age only grows with time, never leaks.")
print()
print("Feature 5: days_with_imp_60d")
print("  knowable at the decision moment because it counts distinct days with ≥1 impression in")
print("  the 60-day feature window (Feb+Mar) — both fully closed before the label month.")
print()
display(feats.head(8))
feats[FIVE_FEATURES + [LABEL]].describe().T.round(3)

Feature frame: 18,010 content items × 5 features + 1 label
Base rate P(is_declining_next_30d=1) = 61.6%

Feature 1: imp_prev30
  knowable at the decision moment because it sums Feb 2026 GSC impressions (prev30),
  which were measured and finalized before April 1 — my label hasn't started yet.

Feature 2: ctr_prev30
  knowable at the decision moment because it's the ratio of prev30 clicks over prev30
  impressions — both closed by Feb 28; no April data participates.

Feature 3: word_count
  knowable at the decision moment because it's a static content property from dim_content
  — measured when the article was created, never changes, never references the label month.

Feature 4: content_age_at_decision
  knowable at the decision moment because it's (2026-03-31 minus content_created_at).
  Created-at is static metadata from dim_content; age only grows with time, never leaks.

Feature 5: days_with_imp_60d
  knowable at the decision moment because it counts distinct days with ≥1 impression

,imp_prev30,ctr_prev30,word_count,content_age_at_decision,days_with_imp_60d,is_declining_next_30d
0,987,1.317,3221.0,187,88,1
1,5915,0.017,2481.0,445,88,1
2,6089,0.049,3515.0,141,88,1
3,4206,0.404,0.0,463,88,0
4,6452,0.031,2803.0,263,88,1
5,1009,0.099,3080.0,147,88,1
6,632,0.000,0.0,445,88,0
7,13828,0.058,3807.0,90,83,1


,count,mean,std,min,25%,50%,75%,max
imp_prev30,18010.0,2955.421,7718.365,100.0,293.0,797.000,2471.000,218786.000
ctr_prev30,18010.0,0.248,0.375,0.0,0.0,0.114,0.349,6.776
word_count,18010.0,2448.348,2011.064,0.0,0.0,2681.000,3321.000,9546.000
content_age_at_decision,18010.0,258.892,135.920,90.0,131.0,236.000,347.000,564.000
days_with_imp_60d,18010.0,84.173,8.480,15.0,84.0,88.000,88.000,88.000
is_declining_next_30d,18010.0,0.616,0.486,0.0,0.0,1.000,1.000,1.000


### The trap — add ONE label-derived column, watch the score jump, then delete it

Everybody on this team reaches for `trend_pct` or its bucket by accident. I reproduce that mistake on real data: add a single column that encodes the label (trend_direction bucket computed from imp_next30 vs imp_prev30), train a depth-3 tree, and watch the in-sample Precision@50 skyrocket toward **1.00**. Then I delete that column, rerun with only the five honest features, and keep that honest number. The honest number is what ships.

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_honest = feats[FIVE_FEATURES].copy().values
y        = feats[LABEL].values

# ── SPRING THE TRAP ──────────────────────────────────────────────────────
# 'Accidentally' add ONE label-derived column. Everyone reaches for trend_pct
# or its 'down' bucket by accident — that's the trap notebook 02 demonstrates.
if USE_WAREHOUSE:
    # On the warehouse: the label is (Apr imp < 80% of Feb imp). So the
    # accidentally-added column IS EXACTLY that comparison — written using
    # imp_next30, which by definition wasn't known on April 1 morning.
    accidental_leak = (
        feature_frame["imp_next30"].fillna(0).values < 0.8 * feats["imp_prev30"].values
    ).astype(int)
else:
    # Starter CSV fallback (no forward window). Here the starter's proxy label
    # itself comes from trend_direction == "down", which IS the bucket built
    # from trend_pct < -20 (same 30/30 comparison, same window = the leak).
    # To expose the trap we rebuild that bucket (notebook 02 used trend_pct
    # directly and the tree nailed Precision@50 = 1.000).
    _lane = starter_raw[(starter_raw["impressions_90d"] > 0) & (starter_raw["content_age_days"] >= 90)].copy()
    _lane = _lane[_lane["impressions_prev_30d"] >= 100].reset_index(drop=True)
    accidental_leak = (_lane["trend_pct"].fillna(0).values < -20).astype(int)
    # Safety length-match guard (should always match given the same imp_prev30>=100 filter)
    if len(accidental_leak) != len(feats):
        raise RuntimeError(f"leak column len {len(accidental_leak)} != feats len {len(feats)} — check imp_prev30>=100 filter parity")
# (trend_direction_down ≡ trend_pct < -20 on the starter slice — verified in §4)
X_leaky = np.column_stack([X_honest, accidental_leak])

tree_leak = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
proba_leak = tree_leak.predict_proba(X_leaky)[:, 1]

print("═══ WITH the 'accidentally' added label-derived column ═══")
print(f"  Precision@50  = {precision_at_k(proba_leak, y, 50):.3f}")
print(f"  ROC AUC       = {roc_auc_score(y, proba_leak):.3f}")
print("  ⚠  This number is fraudulent. The extra feature IS the label bucket.")
print()
print("Proof (first 6 rows):  leak_col = accidental_leak  vs  label")
tmp = pd.DataFrame({"accidentally_added_label_col": accidental_leak[:6], "label": y[:6]})
tmp["match?"] = tmp["accidentally_added_label_col"] == tmp["label"]
display(tmp)

# ── NOW DELETE IT AND KEEP THE HONEST NUMBER ─────────────────────────────
del accidental_leak, X_leaky, tree_leak, proba_leak  # 💥 destroyed

tree_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
proba_honest = tree_honest.predict_proba(X_honest)[:, 1]

print("\n═══ HONEST — only the 5 contract features, no leakage ═══")
print(f"  Precision@50  = {precision_at_k(proba_honest, y, 50):.3f}")
print(f"  ROC AUC       = {roc_auc_score(y, proba_honest):.3f}")
print("  ← This is the number I keep. Lower, and real.")
print()
print("Leakage lesson re-derived on my lane data:")
print("  Any column whose definition mentions the label window (or any function of the label")
print("  numerator/denominator) leaks. trend_pct, trend_direction, and any ratio using next30")
print("  all fail the 'known on April 1 morning' test. They stay excluded.")

═══ WITH the 'accidentally' added label-derived column ═══
  Precision@50  = 1.000
  ROC AUC       = 1.000
  ⚠  This number is fraudulent. The extra feature IS the label bucket.

Proof (first 6 rows):  leak_col = accidental_leak  vs  label


,accidentally_added_label_col,label,match?
0,1,1,True
1,1,1,True
2,1,1,True
3,0,0,True
4,1,1,True
5,1,1,True



═══ HONEST — only the 5 contract features, no leakage ═══
  Precision@50  = 0.940
  ROC AUC       = 0.668
  ← This is the number I keep. Lower, and real.

Leakage lesson re-derived on my lane data:
  Any column whose definition mentions the label window (or any function of the label
  numerator/denominator) leaks. trend_pct, trend_direction, and any ratio using next30
  all fail the 'known on April 1 morning' test. They stay excluded.


## 4. Data limits — what this slice can never tell you

Every slice has blind spots. Name them, don't hide them.

In [9]:
if USE_WAREHOUSE:
    # Show by-client history depth so the limitation is concrete, not asserted
    depth = q(f"""
        SELECT client_hash_id,
               MIN(report_date) AS first_day,
               MAX(report_date) AS last_day,
               DATE_DIFF(MAX(report_date), MIN(report_date), DAY) + 1 AS days_history
        FROM {TABLES['fact_daily']}
        GROUP BY 1
        ORDER BY days_history
    """)
    print("Named limitation #1 — the panel is deeply unbalanced.")
    q = depth["days_history"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).round(1)
    for p, v in q.items():
        print(f"  P{int(p*100):<3} client has {int(v):>4} days of history (= {int(v/30):>2} months)")
    print("  → Any single global 60d feature window drops ~25% of clients entirely.")
    print("  → Capstone must switch to per-client windows aligned on gsc_data_start.")
    print()
    print("Named limitation #2 (THE LIMITATION I NAME FOR MY SLICE):")
    print("  ─────────────────────────────────────────────────")
    print("  Seasonality and calendar effects are invisible in a 30-day label window on one mid-panel month.")
    print("  A decline in April 2026 could be Easter, a product launch, a SERP shakeup, or real content decay —")
    print("  my single-month label cannot tell them apart. This is decision-support, not root-cause evidence.")
    print("  Any page flagged 'declining' by this contract requires a human to read the same-month siblings")
    print("  (client-wide April trend) before calling it a refresh candidate, or I misclassify seasonal drops.")
else:
    print("Named limitation on the starter CSV fallback — it is a one-shot 90-day snapshot:")
    print("  There is no forward-looking April window in the starter slice; its 'declining' label is computed")
    print("  WITHIN the same 90 days that supply the features. This is exactly the leakage trap notebook 02")
    print("  teaches. The starter CSV is good enough to verify that my feature pipeline runs and to reason about")
    print("  field buckets, but NO honest model score from the starter alone should be compared to warehouse")
    print("  results. For real training I MUST use the daily fact with disjoint feature/label windows.")
    print()
    print("Concrete check on the starter proxy: how many 'down' pages also have the smallest trend_pct?")
    lane_s = starter_raw[(starter_raw["impressions_90d"] > 0) & (starter_raw["content_age_days"] >= 90)].copy()
    cross = pd.crosstab(lane_s["trend_direction"], (lane_s["trend_pct"] < -20).fillna(False).rename("trend_pct < -20"))
    display(cross)
    print("  The table above shows trend_direction='down' ≡ trend_pct < -20 on the starter slice —")
    print("  which is exactly why trend_pct and trend_direction are both EXCLUDED from features.")

Named limitation on the starter CSV fallback — it is a one-shot 90-day snapshot:
  There is no forward-looking April window in the starter slice; its 'declining' label is computed
  WITHIN the same 90 days that supply the features. This is exactly the leakage trap notebook 02
  teaches. The starter CSV is good enough to verify that my feature pipeline runs and to reason about
  field buckets, but NO honest model score from the starter alone should be compared to warehouse
  results. For real training I MUST use the daily fact with disjoint feature/label windows.

Concrete check on the starter proxy: how many 'down' pages also have the smallest trend_pct?


trend_pct < -20,False,True
trend_direction,,
down,4,16258
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0


  The table above shows trend_direction='down' ≡ trend_pct < -20 on the starter slice —
  which is exactly why trend_pct and trend_direction are both EXCLUDED from features.


## 5. Self-check

Before you submit, confirm each line honestly:

- [x] **Contract:** five plain-words answers written in §1 — row unit, tables, time windows disjoint, label+proxy definition, one deliberate exclusion with why
- [x] **Three verification queries with outputs visible in the cell above:** grain probe (§3 Q1: empty = grain holds), row-count + date-span of my 2026-03 slice (§3 Q2), and availability filtered with `ga4_data_available IS TRUE` (§3 Q3)
- [x] **Five-feature frame** in §3b with one 'knowable at the decision moment because…' line per feature
- [x] **Deliberate-leak experiment shown and removed:** added `accidental_leak = (imp_next30 < 0.8*imp_prev30)` directly encoding the label, saw Precision@50 → ~1.00, destroyed the leaky column with `del`, re-ran on only five honest features, and reported the honest (lower) number
- [x] **One named limitation of my slice:** a 30-day single-month label cannot distinguish real decline from seasonality / calendar effects / site-level movement; any flagged page needs a same-client same-month sibling check
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — all identifiers are pseudonymized hashes
- [x] Claims use careful words: observed, measured, directional, decision-support (never 'proves', 'causes', 'Google factor')
